# Part A

In [4]:
import os
import subprocess
import sys

USER = 'JasmineLee7'
REPO = 'movie-rec'

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', f'https://github.com/{USER}/{REPO}.git', REPO],
        check=True)

sys.path.insert(0, f'{REPO}/src')
from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline

This code connects to GitHub to clone or update the project repository. It then updates the system path so Python can locate and import the custom evaluation and baseline functions saved during Milestone 2.

In [5]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists('ml-100k'):
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
        'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['user', 'item', 'rating', 'time'])

train, test = split_ratings(ratings, seed=0)
fit, val = split_ratings(train, seed=1)

print("fit:", len(fit), "validation:", len(val), "test:", len(test))

fit: 81200 validation: 8832 test: 9968


This code downloads and extracts the MovieLens 100k dataset if it is not already present. It reads the ratings into a Pandas DataFrame and splits the data twice using a fixed seed: first reserving a hidden test set for final evaluation, and then splitting the rest into a fitting set (81,200 ratings) and a validation set (8,832 ratings) for tuning the models.

## Part B: Set up a fair validation comparison

In [ ]:
R_fit = to_matrix(fit)
known_fit = ~np.isnan(R_fit)
val_truth = val["rating"].to_numpy()
val_u = val["user"].to_numpy() - 1
val_i = val["item"].to_numpy() - 1

def score_at(P, rows, cols, truth):
    """RMSE for selected entries of a prediction matrix."""
    pred = np.clip(P[rows, cols], 1, 5)
    return rmse(pred, truth)

This code creates a user-by-movie matrix using only the fitting ratings. `known_fit` marks entries that contain an observed fitting rating. Validation and test ratings stay out of this matrix so they cannot influence the models. `val_truth` stores the actual validation ratings, and `val_u` and `val_i` store the matching user and movie positions. We subtract 1 because MovieLens IDs start at 1, while Python positions start at 0. The `score_at` function selects the predictions for those user–movie pairs, clips them to the 1–5 star range, and compares them with the actual ratings using RMSE. A lower validation RMSE means more accurate predictions on these held-out ratings; later sections use this score to choose model settings.